# Tutorial: Fluxo do Paciente do Cadastro à Alta

**Público:** alunos que já sabem modelar uma fila isolada e agora precisam enxergar fluxo ponta a ponta.

**Pré-requisitos:** noções de `Resource` e `timeout`.

**Objetivos de aprendizagem:**

- encadear várias etapas em um mesmo processo;
- entender o efeito de retrabalho no sistema;
- perceber que o gargalo pode aparecer no retorno, e não apenas no exame.


## Roteiro

1. Ler o fluxo completo.
2. Identificar os recursos do sistema.
3. Modelar o processo do paciente.
4. Rodar a simulação.
5. Analisar onde o lead time cresce.


In [1]:
from __future__ import annotations

import simpy

print(f"Versão do SimPy: {simpy.__version__}")

Versão do SimPy: 4.1.1


## 1. Cenário

Todo paciente passa por:

- cadastro;
- consulta inicial.

Alguns pacientes ainda passam por:

- laboratório;
- retorno ao médico.

Esse retorno é a parte mais importante do exemplo, porque ele mostra que um paciente pode **voltar a competir por recurso** depois de já ter sido atendido uma vez.


## 2. Conceito fundamental: fluxo não é sempre linear

Em muitos exemplos introdutórios, o paciente faz A -> B -> C e termina.

Na prática, sistemas de saúde têm muito:

- retorno;
- reavaliação;
- exame complementar;
- nova fila depois do exame.

É justamente esse tipo de retrabalho que alonga lead time e muda a percepção do gargalo.


In [2]:
PACIENTES = [
    ("P201", 0, 6, 4, True),
    ("P202", 1, 5, 0, False),
    ("P203", 3, 7, 5, True),
    ("P204", 5, 4, 0, False),
]

PACIENTES

[('P201', 0, 6, 4, True),
 ('P202', 1, 5, 0, False),
 ('P203', 3, 7, 5, True),
 ('P204', 5, 4, 0, False)]

### Estrutura dos dados

Cada tupla contém:

- nome;
- chegada;
- duração da consulta inicial;
- tempo de laboratório;
- indicador se precisa retornar ao médico.


In [3]:
def fluxo_paciente(
    env, nome, chegada, consulta_inicial, lab, retorna, cadastro, medico, laboratorio
):
    yield env.timeout(chegada)
    inicio = env.now
    print(f"{env.now:02.0f} min | {nome} chega")

    with cadastro.request() as req_cadastro:
        yield req_cadastro
        yield env.timeout(2)
        print(f"{env.now:02.0f} min | {nome} conclui cadastro")

    with medico.request() as req_medico:
        yield req_medico
        yield env.timeout(consulta_inicial)
        print(f"{env.now:02.0f} min | {nome} conclui consulta inicial")

    if retorna:
        with laboratorio.request() as req_lab:
            yield req_lab
            yield env.timeout(lab)
            print(f"{env.now:02.0f} min | {nome} conclui laboratório")

        with medico.request() as req_retorno:
            yield req_retorno
            yield env.timeout(2)
            print(f"{env.now:02.0f} min | {nome} conclui retorno médico")

    print(
        f"{env.now:02.0f} min | {nome} recebe alta | lead time={env.now - inicio:02.0f} min"
    )

## 3. Onde está a ideia principal?

A ideia principal está no bloco:

- consulta inicial;
- exame;
- retorno ao médico.

Isso cria uma segunda disputa pelo consultório.

Em outras palavras, o médico não atende apenas a fila inicial. Ele também recebe **pacientes que voltam**.


In [4]:
def executar_simulacao():
    env = simpy.Environment()
    cadastro = simpy.Resource(env, capacity=1)
    medico = simpy.Resource(env, capacity=1)
    laboratorio = simpy.Resource(env, capacity=1)

    for dados in PACIENTES:
        env.process(fluxo_paciente(env, *dados, cadastro, medico, laboratorio))

    env.run()


executar_simulacao()

00 min | P201 chega
01 min | P202 chega
02 min | P201 conclui cadastro
03 min | P203 chega
04 min | P202 conclui cadastro
05 min | P204 chega
06 min | P203 conclui cadastro
08 min | P201 conclui consulta inicial
08 min | P204 conclui cadastro
12 min | P201 conclui laboratório
13 min | P202 conclui consulta inicial
13 min | P202 recebe alta | lead time=12 min
20 min | P203 conclui consulta inicial
24 min | P204 conclui consulta inicial
24 min | P204 recebe alta | lead time=19 min
25 min | P203 conclui laboratório
26 min | P201 conclui retorno médico
26 min | P201 recebe alta | lead time=26 min
28 min | P203 conclui retorno médico
28 min | P203 recebe alta | lead time=25 min


## 4. O que observar na saída

Pontos principais:

- pacientes sem retorno saem mais rápido;
- o laboratório não é o único ponto de espera;
- o retorno ao médico cria nova fila e aumenta o lead time.

Esse é um aprendizado muito útil para operações reais:

**o gargalo percebido pelo paciente pode nascer do retrabalho, não apenas da etapa principal**.


## 5. Erro comum

Um erro frequente é medir só a consulta inicial e esquecer o tempo até a alta.

Em operação, o indicador relevante muitas vezes é o **lead time total**, porque ele captura a experiência ponta a ponta.


## 6. Exercícios

1. Faça todos os pacientes retornarem ao médico. O que acontece com o lead time?
2. Aumente a capacidade do laboratório para `2` e compare.
3. Explique por que `P202` sai antes de `P201`, mesmo chegando depois.


In [5]:
# Espaço para resposta:
# - altere PACIENTES;
# - altere a capacidade dos recursos;
# - rode novamente a simulação e compare as linhas finais.

## 7. Extensão sugerida

Uma evolução natural deste notebook é separar:

- consulta inicial;
- exame;
- laudo;
- retorno;
- observação.

Aí você começa a montar um fluxo realmente próximo de um gêmeo digital simples.
